# vecsearch OpenCL vs CUDA on a Colab T4
Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> Run all (about 25 minutes).
Builds vecsearch with both CUDA and OpenCL, runs the OpenCL tests on the T4, then times the same exact k-NN search (1M x 384, k=10) through the CUDA kernels and the OpenCL port on the same GPU, to measure what portability costs. Downloads `opencl_t4_results.zip` at the end.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!lscpu | grep -E 'Model name|^CPU\(s\)|avx2' | cut -c1-120
!free -g | head -2

OpenCL on NVIDIA: the driver ships the OpenCL library (`libnvidia-opencl.so.1`) but Colab doesn't register it with the ICD loader, so do that by hand and check `clinfo` sees the T4.

In [ ]:
!apt-get install -qq -y ocl-icd-libopencl1 ocl-icd-opencl-dev opencl-headers clinfo > /dev/null
!find / -name 'libnvidia-opencl*' 2>/dev/null | head
import glob, os
libs = glob.glob('/usr/lib*/**/libnvidia-opencl.so*', recursive=True) + glob.glob('/usr/lib*/libnvidia-opencl.so*')
print('nvidia opencl libs:', libs)
os.makedirs('/etc/OpenCL/vendors', exist_ok=True)
if not os.path.exists('/etc/OpenCL/vendors/nvidia.icd'):
    open('/etc/OpenCL/vendors/nvidia.icd', 'w').write((libs[0] if libs else 'libnvidia-opencl.so.1') + '
')
!cat /etc/OpenCL/vendors/nvidia.icd
!clinfo -l
!clinfo | grep -Ei 'platform name|device name|device version|opencl c version|sub-?group|max compute units|max memory allocation|global memory size|local memory size' | cut -c1-140

In [ ]:
BRANCH = 'opencl'
!rm -rf vecsearch && git clone -q -b {BRANCH} https://github.com/Manojkaipu/vecsearch
%cd vecsearch
!git log --oneline -1
!pip install -q faiss-gpu-cu12 matplotlib pytest pyopencl
!CMAKE_ARGS='-DVECSEARCH_BUILD_CUDA=ON -DVECSEARCH_BUILD_OPENCL=ON' pip install -q . 2>&1 | tail -5
import vecsearch as vs
print('cuda available:', vs.cuda_available(), '| opencl available:', vs.opencl_available())
for d in vs.opencl_devices(): print(d)

Correctness on the T4: the CUDA tests and the OpenCL tests (both compare against the CPU index).

In [ ]:
!python -m pytest tests/python/test_gpu.py tests/python/test_opencl.py -q 2>&1 | tail -8

Ceilings the roofline table uses: streaming read bandwidth and FP32 FMA throughput, measured with plain OpenCL kernels on this T4.

In [ ]:
!mkdir -p results/opencl
!python bench/opencl_roofline.py --device NVIDIA --out results/opencl/ceilings_t4.json

Kernel by kernel: the same three distance kernels in CUDA and OpenCL.

In [ ]:
!python bench/gpu_bench.py --backends cuda-naive,cuda-skinny,cuda-tiled,opencl-naive,opencl-skinny,opencl-tiled --opencl-device NVIDIA --batches 1,4,8,16,32,128,1024 --max-batch cuda-skinny=128,opencl-skinny=128 --check 64 --out results/opencl/colab_t4_methods.csv

Main comparison, the CUDA README's sweep: 1M vectors, batch 1 to 4,096.

In [ ]:
!python bench/gpu_bench.py --backends cuda,opencl,opencl-nosg --opencl-device NVIDIA --out results/opencl/colab_t4.csv

In [ ]:
!python bench/roofline.py results/opencl/colab_t4.csv --ceilings results/opencl/ceilings_t4.json --out results/opencl/colab_t4_roofline.md
!python bench/plot_gpu.py results/opencl/colab_t4.csv --out results/opencl/colab_t4_qps.png
!python bench/plot_gpu.py results/opencl/colab_t4_methods.csv --out results/opencl/colab_t4_methods.png
!zip -qr /content/opencl_t4_results.zip results/opencl
from google.colab import files; files.download('/content/opencl_t4_results.zip')